# FridgeChef: AI-Powered Recipe Generator
**Author:** Triet Le  
**Course:** NLP / AI 2026  

FridgeChef is an AI-powered application designed to help users generate recipes based on ingredients currently available. It utilizes FastAPI for the backend, PostgreSQL for data storage, and OpenAI's GPT models for intelligent recipe generation.

## 2. Database Models (SQLAlchemy)
Defines the user and recipe structure for PostgreSQL storage.

In [27]:
# app/models.py
from sqlalchemy import Column, Integer, String, Text, ForeignKey, JSON
from sqlalchemy.orm import relationship, declarative_base

Base = declarative_base()

class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True, index=True)
    email = Column(String, unique=True, index=True)
    hashed_password = Column(String)
    dietary_preferences = Column(JSON, nullable=True) # e.g., ["vegan", "keto"]

    recipes = relationship("SavedRecipe", back_populates="owner")

class SavedRecipe(Base):
    __tablename__ = "saved_recipes"
    id = Column(Integer, primary_key=True, index=True)
    title = Column(String)
    ingredients_used = Column(JSON)
    instructions = Column(Text)
    user_id = Column(Integer, ForeignKey("users.id"))

    owner = relationship("User", back_populates="recipes")

## 3. AI Service Logic (GPT Integration)
Handles interaction with OpenAI's API to generate recipes while preventing hallucinations.

In [28]:
# app/services.py
import openai
import os

# Configure OpenAI API
# openai.api_key = os.getenv("OPENAI_API_KEY")

async def generate_refined_recipe(ingredients: str, preference: str):
    """
    Generates a structured recipe while mitigating hallucinations using
    JSON mode and strict system prompt constraints.
    """
    system_prompt = (
        "You are the FridgeChef AI. You generate recipes based ONLY on the provided ingredients. "
        "STRICT CONSTRAINT: If a dietary preference (e.g., Vegan, Keto) is specified, verify "
        "no prohibited ingredients are used. Return ONLY a JSON object with 'title', "
        "'ingredients', and 'steps'."
    )

    # Example API Call (needs valid API key to run)
    # response = await openai.ChatCompletion.acreate(
    #     model="gpt-4-turbo-preview",
    #     messages=[
    #         {"role": "system", "content": system_prompt},
    #         {"role": "user", "content": f"Ingredients: {ingredients}. Preference: {preference}"}
    #     ],
    #     response_format={"type": "json_object"}
    # )
    # return response.choices[0].message.content
    return {"title": "Recipe Placeholder", "steps": ["Step 1..."]}

## 4. Security & JWT Authentication
Implements password hashing and token generation for secure user access.

In [29]:
# app/auth.py
!pip install python-jose passlib
from jose import jwt
from datetime import datetime, timedelta
from passlib.context import CryptContext

SECRET_KEY = "your-secret-key"
ALGORITHM = "HS256"
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(password: str):
    return pwd_context.hash(password)

def verify_password(plain_password, hashed_password):
    return pwd_context.verify(plain_password, hashed_password)

def create_access_token(data: dict):
    to_encode = data.copy()
    expire = datetime.utcnow() + timedelta(minutes=30)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

## 5. FastAPI Routes
API endpoints for generating recipes and detecting ingredients from images.

In [30]:
# app/api/recipes.py
from fastapi import APIRouter, Depends, File, UploadFile

router = APIRouter(prefix="/recipes", tags=["Recipes"])

@router.post("/generate")
async def create_recipe(ingredients: str, preference: str):
    # Calls the AI service logic
    return {"message": "Recipe logic triggered"}

@router.post("/vision/detect")
async def detect_ingredients(file: UploadFile = File(...)):
    # Placeholder for Phase 2 Computer Vision module
    return {"detected_ingredients": ["tomato", "egg", "spinach"]}

## 6. Unit Testing (Pytest)
Validation logic to ensure dietary filters and response times meet TRD requirements.

In [31]:
# tests/test_logic.py
import pytest
import time

# @pytest.mark.asyncio
async def test_dietary_filter_vegan_logic(recipe_text):
    prohibited = ["egg", "milk", "butter", "meat"]
    for item in prohibited:
        assert item not in recipe_text.lower(), f"Prohibited item {item} found!"

def test_latency_requirement(start_time):
    latency = time.time() - start_time
    assert latency < 3.0, "System failed to meet 3-second latency requirement"

## 7. Deployment Configuration
Containerization strategy for AWS/GCP cloud deployment.

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```